## OCR with Qwen3-VL

This notebook showcases Qwen3-VL's OCR capabilities, including text extraction and recognition from images. See how it accurately captures and interprets text content, even in complex scenarios.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
# Required: OPENAI_API_KEY, OPENAI_BASE_URL, MODEL_ID
load_dotenv()

# Set OpenAI API credentials (required for OpenAI-compatible server)
API_KEY = os.environ["OPENAI_API_KEY"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
MODEL_ID = os.environ["MODEL_ID"]

print("✓ Environment variables loaded successfully")
print(f"  API_KEY: {API_KEY[:8]}..." if API_KEY else "  API_KEY: (empty)")
print(f"  BASE_URL: {BASE_URL}")
print(f"  MODEL_ID: {MODEL_ID}")


### \[Setup\]

Load visualization utils.

In [ ]:
%pip install git+https://github.com/huggingface/transformers
%pip install qwen-vl-utils
%pip install openai

### Plotting Util

In [ ]:


import json
import random
import io
import ast
from PIL import Image, ImageDraw, ImageFont
from PIL import ImageColor
from IPython.display import Markdown, display
from openai import OpenAI
import os
import base64


def plot_text_bounding_boxes(image_path, bounding_boxes):
    """
    Plots bounding boxes on an image with markers for each a name, using PIL, normalized coordinates, and different colors.

    Args:
        image_path: The path to the image file.
        bounding_boxes: A list of bounding boxes containing the name of the object
         and their positions in normalized [y1 x1 y2 x2] format.
    """

    # Load the image
    img = Image.open(image_path)
    width, height = img.size
    print(img.size)
    # Create a drawing object
    draw = ImageDraw.Draw(img)

    # Parsing out the markdown fencing
    bounding_boxes = parse_json(bounding_boxes)

    font = ImageFont.truetype("NotoSansCJK-Regular.ttc", size=10)

    # Iterate over the bounding boxes
    for i, bounding_box in enumerate(ast.literal_eval(bounding_boxes)):
      color = 'green'

      # Convert normalized coordinates to absolute coordinates
      abs_y1 = int(bounding_box["bbox_2d"][1]/999 * height)
      abs_x1 = int(bounding_box["bbox_2d"][0]/999 * width)
      abs_y2 = int(bounding_box["bbox_2d"][3]/999 * height)
      abs_x2 = int(bounding_box["bbox_2d"][2]/999 * width)

      if abs_x1 > abs_x2:
        abs_x1, abs_x2 = abs_x2, abs_x1

      if abs_y1 > abs_y2:
        abs_y1, abs_y2 = abs_y2, abs_y1

      # Draw the bounding box
      draw.rectangle(
          ((abs_x1, abs_y1), (abs_x2, abs_y2)), outline=color, width=1
      )

      # Draw the text
      if "text_content" in bounding_box:
        draw.text((abs_x1, abs_y2), bounding_box["text_content"], fill=color, font=font)

    # Display the image
    img.show()

# @title Parsing JSON output
def parse_json(json_output):
    # Parsing out the markdown fencing
    lines = json_output.splitlines()
    for i, line in enumerate(lines):
        if line == "```json":
            json_output = "\n".join(lines[i+1:])  # Remove everything before "```json"
            json_output = json_output.split("```")[0]  # Remove everything after the closing "```"
            break  # Exit the loop once "```json" is found
    return json_output


#  base 64 编码格式
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# @title inference function with API
def inference_with_api(image_path, prompt, sys_prompt="You are a helpful assistant.", model_id=MODEL_ID, min_pixels=512*32*32, max_pixels=2048*32*32):
    base64_image = encode_image(image_path)
    client = OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL
    )


    messages=[
        # skip the system prompt
        # {
        #     "role": "system",
        #     "content": [{"type":"text","text": sys_prompt}]},
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "min_pixels": min_pixels,
                    "max_pixels": max_pixels,
                    # Pass in BASE64 image data. Note that the image format (i.e., image/{format}) must match the Content Type in the list of supported images. "f" is the method for string formatting.
                    # PNG image:  f"data:image/png;base64,{base64_image}"
                    # JPEG image: f"data:image/jpeg;base64,{base64_image}"
                    # WEBP image: f"data:image/webp;base64,{base64_image}"
                    "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]
    completion = client.chat.completions.create(
        model = model_id,
        messages = messages,
       
    )
    return completion.choices[0].message.content

##### 1. Full-page OCR for English text:

In [ ]:
image_path = "./assets/ocr/ocr_example2.jpg"
prompt = "Read all the text in the image."

image = Image.open(image_path)
display(image.resize((400,600)))


response = inference_with_api(image_path, prompt)
print(response)


##### 2. Full Page OCR for Multilingual text

In [ ]:
image_path = "./assets/ocr/ocr_example6.jpg"
prompt = "Please output only the text content from the image without any additional descriptions or formatting."

image = Image.open(image_path)
display(image.resize((400,400)))

response = inference_with_api(image_path, prompt)
print(response)

In [ ]:
image_path = "./assets/ocr/ocr_example5.jpg"
#prompt = "请识别出图中所有的文字。"
prompt = "Please translate the text to spanish"

image = Image.open(image_path)
display(image.resize((512,512)))


response = inference_with_api(image_path, prompt)
print(response)

#### 3. Text Spotting

Text spotting is to  output the localization and content of all appeared text in the image. Note that for Qwen3-VL series, the localization of text is in the format of relative coordinates ranging from 0 to 999.

In [ ]:
image_path = "./assets/ocr/ocr_example3.jpg"
prompt = "Spotting all the text in the image with line-level, and output in JSON format as [{'bbox_2d': [x1, y1, x2, y2], 'text_content': 'text'}, ...]."

min_pixels = 512*32*32
max_pixels = 2048*32*32
image = Image.open(image_path)
width, height = image.size
response = inference_with_api(image_path, prompt, min_pixels=min_pixels, max_pixels=max_pixels)
display(Markdown(response))
plot_text_bounding_boxes(image_path, response)




In [ ]:
image_path = "./assets/ocr/ocr_example2.jpg"
prompt = "Spotting all the text in the image with word-level, and output in JSON format as [{'bbox_2d': [x1, y1, x2, y2], 'text_content': 'text'}, ...]."

min_pixels = 512*32*32
max_pixels = 2048*32*32
image = Image.open(image_path)
width, height = image.size
response = inference_with_api(image_path, prompt, min_pixels=min_pixels, max_pixels=max_pixels)
display(Markdown(response))
plot_text_bounding_boxes(image_path, response)

#### 4. Visual Information Extraction

Visual information Extraction with given keys.

In [ ]:
image_path = "./assets/ocr/ocr_example3.jpg"
prompt = "Extract the key-value information in the format:{\"company\": \"\", \"date\": \"\", \"address\": \"\", \"total\": \"\"}"

image = Image.open(image_path)
display(image.resize((300,500)))

response = inference_with_api(image_path, prompt)
display(Markdown(response))

In [ ]:
image_path = "./assets/ocr/ocr_example4.jpg"
prompt = "提取图中的：['发票代码','发票号码','到站','燃油费','票价','乘车日期','开车时间','车次','座号']，并且按照json格式输出。"

image = Image.open(image_path)
display(image.resize((400,400)))


response = inference_with_api(image_path, prompt)
display(Markdown(response))
